# 01 - Exploratory Data Analysis (EDA)

Project: **sample_ml_project**

This notebook provides a standardized structure for initial data exploration. It is designed to be dependency-free by default, using Python's standard library, but you can easily integrate `pandas`, `matplotlib`, and `seaborn` for more advanced analysis.

## 1. Setup

In [ ]:
import json
import csv
from pathlib import Path
from collections import Counter

# Add src to path if needed
import sys
import os
if os.getcwd().endswith('notebooks'):
    sys.path.append('..')

from src.sample_ml_project.config import load_config, project_root

## 2. Load Configuration

In [ ]:
config = load_config()
raw_data_path = project_root() / config['data']['raw_path']
target_column = config['target'].get('column')
eda_config = config.get('eda', {})

print(f"Project: {config['project']['name']}")
print(f"Task: {config['project']['task']}")
print(f"Dataset: {raw_data_path}")
print(f"Target: {target_column}")

## 3. Load Data

By default, we use the standard library `csv` module to keep the project lightweight.

In [ ]:
def load_data(path):
    if not path.exists():
        print(f"Warning: Dataset not found at {path}")
        return []
    
    with open(path, 'r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        return list(reader)

data = load_data(raw_data_path)
print(f"Rows loaded: {len(data)}")

### Optional: Use Pandas
If you have `pandas` installed, you can use it for better performance and easier analysis:
```python
import pandas as pd
df = pd.read_csv(raw_data_path)
df.head()
```

## 4. Basic Inspection

### Shape and Columns

In [ ]:
if data:
    columns = list(data[0].keys())
    print(f"Number of columns: {len(columns)}")
    print("Columns:", columns)
else:
    print("No data to inspect.")

### Missing Values

In [ ]:
if data:
    missing = {col: 0 for col in columns}
    for row in data:
        for col in columns:
            if row[col] in (None, '', 'NaN', 'null'):
                missing[col] += 1
    
    for col, count in missing.items():
        if count > 0:
            pct = (count / len(data)) * 100
            print(f"{col}: {count} ({pct:.2f}%)")
    if not any(missing.values()):
        print("No missing values detected (in basic string checks).")

### Duplicated Rows

In [ ]:
if data:
    # Note: converting dict to tuple for hashability
    unique_rows = set(tuple(row.items()) for row in data)
    dupes = len(data) - len(unique_rows)
    print(f"Duplicated rows: {dupes} ({ (dupes/len(data))*100 if data else 0 :.2f}%)")

## 5. Target Distribution

Analysis of the variable we want to predict.

In [ ]:
if data and target_column and target_column in columns:
    target_vals = [row[target_column] for row in data]
    counts = Counter(target_vals)
    print(f"Target: {target_column}")
    for val, count in counts.most_common():
        pct = (count / len(data)) * 100
        print(f"{val}: {count} ({pct:.2f}%)")
elif not target_column:
    print("No target column configured.")
else:
    print(f"Target column '{target_column}' not found in dataset.")

## 6. Numerical and Categorical Summaries

In [ ]:
def is_numeric(val):
    try:
        float(val)
        return True
    except (ValueError, TypeError):
        return False

if data:
    # Guessing types from the first 100 rows
    numeric_cols = eda_config.get('numeric_columns', [])
    if not numeric_cols:
        for col in columns:
            if all(is_numeric(row[col]) for row in data[:100] if row[col]):
                numeric_cols.append(col)
    
    print(f"Detected/Configured numeric columns: {numeric_cols}")
    
    for col in numeric_cols:
        vals = [float(row[col]) for row in data if is_numeric(row[col])]
        if vals:
            print(f"\n-- {col} --")
            print(f"Min: {min(vals)}")
            print(f"Max: {max(vals)}")
            print(f"Mean: {sum(vals)/len(vals):.2f}")
            
    categorical_cols = eda_config.get('categorical_columns', [])
    if not categorical_cols:
        for col in columns:
            if col not in numeric_cols and col != target_column and col not in eda_config.get('id_columns', []):
                categorical_cols.append(col)
    
    print(f"\nDetected/Configured categorical columns: {categorical_cols}")
    
    for col in categorical_cols:
        print(f"\n-- {col} --")
        vals = [row[col] for row in data]
        counts = Counter(vals)
        for val, count in counts.most_common(10):
            pct = (count / len(data)) * 100
            print(f"{val}: {count} ({pct:.2f}%)")
        if len(counts) > 10:
            print(f"... and {len(counts) - 10} more unique values")

## 7. Data Quality & Leakage

Check columns configured in `configs/config.json`:
- **ID Columns:** Columns that are unique identifiers and should not be used as features.
- **Date Columns:** Columns that represent time.
- **Suspected Leakage:** Columns that might contain information from the future.

Leakage happens when information from the future or information that won't be available at prediction time is included in training features.

In [ ]:
print("ID columns:", eda_config.get('id_columns', []))
print("Date columns:", eda_config.get('date_columns', []))
print("Suspected leakage columns:", eda_config.get('suspected_leakage_columns', []))

## 8. Feature Ideas

Write down your hypotheses here:
1. ...
2. ...
3. ...

## 9. Next Steps

1. Update `configs/config.json` with identified column types.
2. Implement features in `src/sample_ml_project/features.py`.
3. Run training script.